# Swap in Popcorn

Consider training code for a small hybrid LM written in plain PyTorch, with alternating Wall Attention and RoPE attention blocks.

In this example, we define the model in `modeling.py` and allow passing functions for kernel calls to demonstrate swapping implementations.

In [1]:
import torch
from modeling import CausalLM, ModelConfig

torch.manual_seed(0)
device = "cuda"
config = ModelConfig()

batch = {
    "input_ids": torch.randint(config.vocab_size, (2, 512), device=device),
    "labels": torch.randint(config.vocab_size, (2, 512), device=device),
}

Such a model combines kernels from multiple third party libraries. Some kernels (such as `rms_norm`) are offered by multiple libraries and it is unclear which one to use here. Furthermore, for different sized models, one kernel implementation may be faster than another. Most modeling code settles for one backend even if faster kernels exist.

In [2]:
!uv pip install flash-linear-attention liger-kernel "git+https://github.com/tilde-research/wall-attention-release.git"

from fla.modules.activations import swiglu
from fla.modules.layernorm import rms_norm
from liger_kernel.transformers.functional import liger_fused_linear_cross_entropy, liger_rope
from wall_attn.training import wall_attn

In [3]:
manual = CausalLM(
    config,
    # FLA takes `bias` as a required positional and defaults eps to 1e-5, so it needs an adapter.
    rms_norm=lambda x, weight, eps=1e-6: rms_norm(x, weight, None, eps=eps),
    rope=liger_rope,
    swiglu=swiglu,
    wall_attn=wall_attn,
    linear_cross_entropy=liger_fused_linear_cross_entropy,
).to(device)

The above code is inconsistent and requires a whole harness to optimize kernels for correct shapes. Trying unsloth's rope instead means another permanent dependency and another signature, and none of it knows which kernel is actually faster at your shapes or which ones are silently wrong in fp16.

Popcorn bounds those same kernels (and more) behind one set of names, with signatures that already match the reference. Each implementation is verified across a large grid to ensure correctness, and the timings from that grid pick the implementation for you.

In [4]:
# Drop the hand-wired names, so the ones below can only have come from Popcorn.
del swiglu, rms_norm, liger_fused_linear_cross_entropy, liger_rope, wall_attn

In [5]:
!uv pip install "popcorn[fla,liger]"

from popcorn.kernels import linear_cross_entropy, rms_norm, rope, swiglu, wall_attn

In [6]:
model = CausalLM(
    config,
    rms_norm=rms_norm,
    rope=rope,
    swiglu=swiglu,
    wall_attn=wall_attn,
    linear_cross_entropy=linear_cross_entropy,
).to(device)
model.load_state_dict(manual.state_dict())

<All keys matched successfully>

No glue, and each op is chosen per shape and dtype from the benchmark reports. Same weights, so the loss and the training step should match what you wired by hand.

In [7]:
from popcorn.bench import compare, report


def train(model):
    def step(**batch):
        loss = model(**batch)
        loss.backward()
        model.zero_grad(set_to_none=True)
        return loss.detach()

    return step


# Triton autotunes on first use, which takes a few minutes on a cold cache. Warm both sides first.
for warm in (train(manual), train(model)):
    warm(**batch)

options = dict(mine="popcorn", reference="hand-wired", assert_rel=0.02)
report(compare(model, manual, batch, backward=False, repeats=3), "loss", **options)
report(compare(train(model), train(manual), batch, backward=False, repeats=2), "train", **options);

loss     hand-wired     25.6 ms -> popcorn   22.7 ms   (1.1x)
         peak         1.35 GB ->      4.02 GB
         err 0.00e+00 on scale 10.5


train    hand-wired     48.7 ms -> popcorn   46.1 ms   (1.1x)
         peak         1.55 GB ->      2.05 GB
         err 0.00e+00 on scale 10.5


As can be seen, popcorn matches the hand-wired model with none of the wiring.

Each op still tells you what it has and lets you pin one when you are debugging.

In [8]:
x = torch.randn(2048, config.hidden_size, device=device)
weight = torch.ones(config.hidden_size, device=device)

dispatched = rms_norm(x, weight)
forced = rms_norm["fla"](x, weight)

rms_norm

rms_norm(x: jaxtyping.Float[Tensor, '... normalized_shape'], weight: jaxtyping.Float[Tensor, 'normalized_shape'], bias: jaxtyping.Float[Tensor, 'normalized_shape'] | None = None, eps: float = 1e-06) -> jaxtyping.Float[Tensor, '... normalized_shape']
  fla
  liger  gates={'bias': typing.Literal[None]}
  quack
  unsloth  fwd-only  gates={'bias': typing.Literal[None]}
  torch